# CMAT 651 · Dataset Freeze Notebook
**Dataset:** WNBA 2025 shot chart (one row = one field-goal attempt)  
**Outcome:** `made_flag` (1 = made, 0 = missed)  
**Seed:** 651 for every random step  

This notebook is run **once**. It cleans the raw file, writes the full clean file (stays on my laptop, not committed), writes a 10,000-row sample (committed), and writes a freeze receipt (committed). Every Data slide uses the clean file, or the sample if the slide says so.

In [ ]:
import hashlib, json, platform, datetime
import numpy as np
import pandas as pd

SEED = 651
RAW = "wnba_shots_2025.csv"        # raw file as downloaded; never edited
CLEAN = "wnba_clean.csv"           # full clean file: STAYS ON MY LAPTOP (gitignore it)
SAMPLE = "wnba_sample_10k.csv"     # 10,000-row sample: committed
RECEIPT = "freeze_receipt.json"         # committed

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)
print("seed", SEED)

python 3.12.3 | pandas 3.0.2 | numpy 2.4.4
seed 651


## 1. Load the raw file and audit it

In [2]:
raw = pd.read_csv(RAW)
raw_sha = sha256(RAW)
print("raw shape:", raw.shape)
print("raw sha256:", raw_sha)
print()
print("missing values per column (raw):")
print(raw.isna().sum()[raw.isna().sum() > 0] if raw.isna().any().any() else "none")
print()
print("exact duplicate rows:", raw.duplicated().sum())
KEY = ["GAME_ID", "GAME_EVENT_ID", "PLAYER_ID"]
print("duplicate (game, event, player) keys:", raw.duplicated(KEY).sum())
print()
print("columns with a single value:", [c for c in raw.columns if raw[c].nunique() == 1])
print()
print("EVENT_TYPE vs SHOT_MADE_FLAG (should be perfectly redundant):")
print(pd.crosstab(raw["EVENT_TYPE"], raw["SHOT_MADE_FLAG"]))

raw shape: (38535, 24)
raw sha256: 12979f93e9d8e903f1fb2a8f9039b0a5d94109a5c8aab178f8a8d6f7b0bd60bf

missing values per column (raw):
none

exact duplicate rows: 0
duplicate (game, event, player) keys: 0

columns with a single value: ['GRID_TYPE', 'SHOT_ATTEMPTED_FLAG']

EVENT_TYPE vs SHOT_MADE_FLAG (should be perfectly redundant):
SHOT_MADE_FLAG      0      1
EVENT_TYPE                  
Made Shot           0  16926
Missed Shot     21609      0


## 2. Cleaning steps (each one has a reason)
| # | Step | Why |
|---|------|-----|
| 1 | Drop exact / key duplicates | A shot recorded twice would double-count |
| 2 | Lowercase column names | Easier to type in later notebooks |
| 3 | Drop constant columns | Zero variance carries no information and breaks standardizing in PCA |
| 4 | Drop `event_type` | Perfectly redundant with `made_flag` |
| 5 | Parse `game_date` | Stored as an integer like 20250517 |
| 6 | Make `seconds_left` and drop minutes/seconds | One clock column instead of two pieces of the same clock |
| 7 | Add `is_overtime` and `is_backcourt` flags | Mark unusual shots **without deleting them** |
| 8 | Sort rows in a fixed order | Same row order every run, so seed 651 gives the same sample |

In [3]:
df = raw.copy()
log = [{"step": "raw", "rows": len(df), "cols": df.shape[1]}]
def record(name):
    log.append({"step": name, "rows": len(df), "cols": df.shape[1]})

# 1. duplicates
df = df.drop_duplicates()
df = df.drop_duplicates(subset=KEY)
record("drop duplicates")

# 2. column names
df.columns = df.columns.str.lower()
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip()
record("lowercase names, strip text")

# 3. constant columns
const_cols = [c for c in df.columns if df[c].nunique() == 1]
print("dropping constant columns:", const_cols)
df = df.drop(columns=const_cols)
record("drop constant columns")

# 4. event_type is the same information as made_flag
assert ((df["event_type"] == "Made Shot") == (df["shot_made_flag"] == 1)).all()
df = df.drop(columns="event_type")
df = df.rename(columns={"shot_made_flag": "made_flag"})
record("drop event_type, rename made_flag")

# 5. dates
df["game_date"] = pd.to_datetime(df["game_date"].astype(str), format="%Y%m%d")
record("parse game_date")

# 6. one clock column
df["seconds_left"] = df["minutes_remaining"] * 60 + df["seconds_remaining"]
df = df.drop(columns=["minutes_remaining", "seconds_remaining"])
record("seconds_left")

# 7. flags (rows are kept)
df["is_overtime"] = (df["period"] >= 5).astype(int)
df["is_backcourt"] = (df["shot_zone_basic"] == "Backcourt").astype(int)
record("flags")

# 8. fixed order
df = df.sort_values(["game_id", "game_event_id", "player_id"]).reset_index(drop=True)
record("sort")

print(pd.DataFrame(log).to_string(index=False))

dropping constant columns: ['grid_type', 'shot_attempted_flag']
                             step  rows  cols
                              raw 38535    24
                  drop duplicates 38535    24
      lowercase names, strip text 38535    24
            drop constant columns 38535    22
drop event_type, rename made_flag 38535    21
                  parse game_date 38535    21
                     seconds_left 38535    20
                            flags 38535    22
                             sort 38535    22


## 3. Checks that must pass before freezing

In [4]:
assert df["made_flag"].isin([0, 1]).all()
assert df["shot_distance"].ge(0).all()
assert df["seconds_left"].between(0, 600).all()
assert df["period"].between(1, 6).all()
assert not df.isna().any().any()
assert not df.duplicated([k.lower() for k in KEY]).any()
# distance column agrees with court coordinates (LOC units are tenths of a foot)
gap = (np.hypot(df["loc_x"], df["loc_y"]) / 10 - df["shot_distance"]).abs()
assert gap.max() <= 2, gap.max()
print("all checks passed")
print()
print("clean shape:", df.shape)
print("columns:", list(df.columns))
print()
print("outcome rate (made_flag mean):", round(df["made_flag"].mean(), 4))
print("games:", df["game_id"].nunique(), "| players:", df["player_id"].nunique(), "| teams:", df["team_name"].nunique())
print("overtime shots:", int(df["is_overtime"].sum()), "| backcourt shots:", int(df["is_backcourt"].sum()))
print()
print(df[["shot_distance", "loc_x", "loc_y", "seconds_left", "period"]].describe().round(2))

all checks passed

clean shape: (38535, 22)
columns: ['game_id', 'game_event_id', 'player_id', 'player_name', 'team_id', 'team_name', 'period', 'action_type', 'shot_type', 'shot_zone_basic', 'shot_zone_area', 'shot_zone_range', 'shot_distance', 'loc_x', 'loc_y', 'made_flag', 'game_date', 'htm', 'vtm', 'seconds_left', 'is_overtime', 'is_backcourt']

outcome rate (made_flag mean): 0.4392
games: 286 | players: 179 | teams: 13
overtime shots: 96 | backcourt shots: 92

       shot_distance     loc_x     loc_y  seconds_left    period
count       38535.00  38535.00  38535.00      38535.00  38535.00
mean           12.98     -3.59     90.55        291.72      2.47
std             9.93    110.05     87.77        172.66      1.13
min             0.00   -249.00    -49.00          0.00      1.00
25%             3.00    -49.00     17.00        142.00      1.00
50%            11.00      0.00     54.00        294.00      2.00
75%            23.00     40.00    165.00        442.00      3.00
max        

## 4. Freeze: write the clean file and the 10,000-row sample

In [5]:
df.to_csv(CLEAN, index=False)
sample = df.sample(n=10_000, random_state=SEED).sort_index()
sample.to_csv(SAMPLE, index=False)

print("clean rows:", len(df), "| sha256:", sha256(CLEAN))
print("sample rows:", len(sample), "| sha256:", sha256(SAMPLE))
print()
print("full vs sample (should be close, not identical):")
print(pd.DataFrame({
    "full": df[["made_flag", "shot_distance", "loc_x", "loc_y", "seconds_left"]].mean(),
    "sample": sample[["made_flag", "shot_distance", "loc_x", "loc_y", "seconds_left"]].mean(),
}).round(4))

clean rows: 38535 | sha256: 6dfadf4db636d0371e66d154a511b5d63ceac5d3b5819d486b74e686c85fe2a7
sample rows: 10000 | sha256: 40e57c59bf13d6031b016816c6cbf3d506b7dac24a36736f890135476d5ba25b

full vs sample (should be close, not identical):
                   full    sample
made_flag        0.4392    0.4374
shot_distance   12.9834   12.9886
loc_x           -3.5944   -3.0103
loc_y           90.5455   90.2644
seconds_left   291.7158  291.3401


In [6]:
receipt = {
    "course": "CMAT 651",
    "dataset": "WNBA 2025 shot chart",
    "outcome": "made_flag",
    "seed": SEED,
    "frozen_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "raw_file": RAW,
    "raw_sha256": raw_sha,
    "raw_rows": int(raw.shape[0]),
    "raw_cols": int(raw.shape[1]),
    "clean_file": CLEAN,
    "clean_sha256": sha256(CLEAN),
    "clean_rows": int(df.shape[0]),
    "clean_cols": int(df.shape[1]),
    "clean_columns": list(df.columns),
    "sample_file": SAMPLE,
    "sample_sha256": sha256(SAMPLE),
    "sample_rows": int(sample.shape[0]),
    "sample_method": "df.sample(n=10000, random_state=651).sort_index()",
    "cleaning_log": log,
    "independent_unit_for_group_bootstrap": "game_id (shots within a game, and within a player, are not independent)",
}
with open(RECEIPT, "w") as f:
    json.dump(receipt, f, indent=2)
print(json.dumps({k: receipt[k] for k in ["seed", "frozen_at", "raw_rows", "clean_rows", "sample_rows", "clean_sha256", "sample_sha256"]}, indent=2))

{
  "seed": 651,
  "frozen_at": "2026-10-25T09:29:18",
  "raw_rows": 38535,
  "clean_rows": 38535,
  "sample_rows": 10000,
  "clean_sha256": "6dfadf4db636d0371e66d154a511b5d63ceac5d3b5819d486b74e686c85fe2a7",
  "sample_sha256": "40e57c59bf13d6031b016816c6cbf3d506b7dac24a36736f890135476d5ba25b"
}


## AI Note

- **What AI helped with:** an AI assistant (Claude) drafted the pandas code in cells 2–7 and the receipt-writing cell.
- **What I checked and can explain:** I can explain every line: why `drop_duplicates(subset=KEY)` uses three columns, why `sort_values` comes before `sample`, why `random_state=651` makes the sample repeatable.
- **Decisions that are mine:** I chose to keep backcourt shots and flag them instead of dropping them because they are reasonable shots within the context that they are taken despite being abnormal. Decided it best to remove SHOT_ATTEMPTED_FLAG since all shots within the dataset is true for this value.
- **What AI did not do:** The words on my slides, my interpretations and my derivations are my own.